<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/08_llm_as_judge_langfuse.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# LLM-as-Judge: Evaluating AI Summaries for Faithfulness

## What you will learn
- Run **format validation checks** on an AI-generated summary (word count, required fields).
- Use an **LLM-as-judge** to evaluate whether a summary is faithful to its source — catching errors that format checks miss.
- Optionally log the full evaluation pipeline to **Langfuse** for tracing and monitoring.

**NLP Task**: Summarisation evaluation — assessing whether an AI-generated summary accurately represents its source document.

**Scenario**: NordWind Energy daily field report → AI-generated summary → Evaluation pipeline.

**Key question**: Can a summary pass all format checks and still be dangerously wrong?

Expected runtime: ~2 minutes (one LLM API call for the judge)
Expected cost: Free-tier Gemini

## Step 1 — Setup & Dependencies

We need the Gemini SDK for the LLM judge call. Langfuse is optional — if you have an account, install the SDK and provide your keys. If not, the notebook runs fine without it.

In [1]:
%pip install -q google-genai langfuse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 1.9 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import textwrap

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print(f'Using Gemini model: {GEMINI_MODEL}')

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

gemini_client = genai.Client(api_key=api_key)
print('Gemini client ready.')

Using Gemini model: gemini-3.1-flash-lite-preview
Gemini client ready.


In [4]:
# --- Langfuse Setup (Optional) ---
# If you have a free Langfuse account (langfuse.com), add your keys as Colab
# secrets: LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY. If you don't, leave them
# blank — the notebook will still run, it just won't log to an external dashboard.

LANGFUSE_ENABLED = False
langfuse = None

try:
    langfuse_public_key = os.getenv('LANGFUSE_PUBLIC_KEY')
    langfuse_secret_key = os.getenv('LANGFUSE_SECRET_KEY')

    if not langfuse_public_key or not langfuse_secret_key:
        try:
            from google.colab import userdata
            langfuse_public_key = langfuse_public_key or userdata.get('LANGFUSE_PUBLIC_KEY')
            langfuse_secret_key = langfuse_secret_key or userdata.get('LANGFUSE_SECRET_KEY')
        except Exception:
            pass

    if langfuse_public_key and langfuse_secret_key:
        os.environ['LANGFUSE_PUBLIC_KEY'] = langfuse_public_key
        os.environ['LANGFUSE_SECRET_KEY'] = langfuse_secret_key
        os.environ['LANGFUSE_BASE_URL'] = os.getenv(
            'LANGFUSE_BASE_URL', 'https://cloud.langfuse.com'
        )

        from langfuse import get_client
        langfuse = get_client()
        LANGFUSE_ENABLED = True
        print('✅ Langfuse enabled — traces will be logged.')
    else:
        print('ℹ️  Langfuse keys not found. Running without Langfuse (results shown inline only).')

except Exception as e:
    print(f'ℹ️  Langfuse setup skipped: {e}')
    print('   Results will be shown inline only.')

✅ Langfuse enabled — traces will be logged.


## Step 2 — The Source Field Report

This is a real-style NordWind Energy daily field report from the Hvide Sande site.
Read it carefully — you'll need to compare the AI summary against it.

In [5]:
SOURCE_REPORT = """NordWind Energy — Daily Field Report
Site: Hvide Sande | Date: 2026-02-17 | Technician: Lars Eriksen

Completed scheduled inspection on turbines NW-023 and NW-024. NW-023 pitch
bearing showed elevated vibration levels (4.2 mm/s RMS) — within tolerance but
trending upward over the past three inspection cycles. Recommend follow-up
inspection within 30 days to assess progression. NW-024 generator brush
inspection nominal — no action required.

Replaced worn CTV mooring line on vessel Havørn. Old line showed visible
fraying at attachment points. Replacement completed by 11:30 and vessel
returned to service for afternoon transfers.

Wind conditions: 12–18 m/s, occasional gusts to 24 m/s. Operations paused for
45 minutes (13:15–14:00) due to wave height exceeding transfer threshold.
Resumed after conditions improved. No personnel safety incidents."""

print(SOURCE_REPORT)
print(f'\n— Source report: {len(SOURCE_REPORT.split())} words')

NordWind Energy — Daily Field Report
Site: Hvide Sande | Date: 2026-02-17 | Technician: Lars Eriksen

Completed scheduled inspection on turbines NW-023 and NW-024. NW-023 pitch
bearing showed elevated vibration levels (4.2 mm/s RMS) — within tolerance but
trending upward over the past three inspection cycles. Recommend follow-up
inspection within 30 days to assess progression. NW-024 generator brush
inspection nominal — no action required.

Replaced worn CTV mooring line on vessel Havørn. Old line showed visible
fraying at attachment points. Replacement completed by 11:30 and vessel
returned to service for afternoon transfers.

Wind conditions: 12–18 m/s, occasional gusts to 24 m/s. Operations paused for
45 minutes (13:15–14:00) due to wave height exceeding transfer threshold.
Resumed after conditions improved. No personnel safety incidents.

— Source report: 122 words


## Step 3 — The AI-Generated Summary (with a planted error)

Below is a pre-written summary. In a real pipeline, this would come from an LLM
call. We use a fixed string so the demo is **reproducible** — everyone sees the
same evaluation result.

Read the summary. Does anything look off?

In [6]:
# This summary was pre-generated. In a real pipeline, it would come from an
# LLM call. We use a fixed string so the demo is reproducible.

AI_SUMMARY = (
    "Daily summary for Hvide Sande, 17 Feb 2026: Routine inspections on "
    "NW-023 and NW-024 completed. NW-023 pitch bearing vibration trending "
    "upward; 30-day follow-up recommended. CTV mooring line replaced on "
    "vessel Havørn. Downtime reduced by 23% compared to previous week. "
    "Operations briefly paused due to wave conditions."
)

print(AI_SUMMARY)
print(f'\n— Summary: {len(AI_SUMMARY.split())} words')

Daily summary for Hvide Sande, 17 Feb 2026: Routine inspections on NW-023 and NW-024 completed. NW-023 pitch bearing vibration trending upward; 30-day follow-up recommended. CTV mooring line replaced on vessel Havørn. Downtime reduced by 23% compared to previous week. Operations briefly paused due to wave conditions.

— Summary: 46 words


## Step 4 — Format Validation Checks

These are basic sanity checks: Is the summary the right length? Does it mention
the turbines and date from the source? These are simple, deterministic checks
that catch formatting problems — but not factual errors.

In [7]:
def check_word_count(summary: str, max_words: int = 60) -> dict:
    """Check if the summary is within the word limit."""
    count = len(summary.split())
    passed = count <= max_words
    return {
        "check": "word_count",
        "passed": passed,
        "detail": f"{count} words (limit: {max_words})",
    }


def check_turbine_ids(summary: str, expected_ids: list[str]) -> dict:
    """Check if at least one expected turbine ID appears in the summary."""
    found = [tid for tid in expected_ids if tid in summary]
    passed = len(found) > 0
    return {
        "check": "turbine_id",
        "passed": passed,
        "detail": f"Found: {', '.join(found)}" if found else "No expected IDs found",
    }


def check_date_present(summary: str, expected_date: str = "2026-02-17") -> dict:
    """Check if the report date appears in any reasonable format."""
    # Check ISO format, European format, and written formats
    date_patterns = [
        expected_date,                  # 2026-02-17
        "17 Feb 2026",                  # written
        "17 February 2026",             # full month
        "Feb 17, 2026",                 # US format
        "February 17, 2026",            # US full
        "17/02/2026",                   # European slash
        "17.02.2026",                   # European dot
    ]
    found = [p for p in date_patterns if p.lower() in summary.lower()]
    passed = len(found) > 0
    return {
        "check": "date_present",
        "passed": passed,
        "detail": f"Matched: {found[0]}" if found else "Date not found in summary",
    }


# --- Run all format checks ---
format_results = [
    check_word_count(AI_SUMMARY),
    check_turbine_ids(AI_SUMMARY, expected_ids=["NW-023", "NW-024"]),
    check_date_present(AI_SUMMARY),
]

print("Format Validation Results")
print("=" * 50)
for r in format_results:
    status = "✅ Pass" if r["passed"] else "❌ Fail"
    print(f"  {r['check']:<20} {status}  ({r['detail']})")

all_format_passed = all(r["passed"] for r in format_results)
print(f"\nAll format checks passed: {all_format_passed}")
print("\nThe summary reads well and meets all structural requirements.")
print("But is everything in it actually true? →")

Format Validation Results
  word_count           ✅ Pass  (46 words (limit: 60))
  turbine_id           ✅ Pass  (Found: NW-023, NW-024)
  date_present         ✅ Pass  (Matched: 17 Feb 2026)

All format checks passed: True

The summary reads well and meets all structural requirements.
But is everything in it actually true? →


## Step 5 — LLM-as-Judge: Faithfulness Check

Now for the evaluation that matters most. We send both the **source report** and
the **AI summary** to a second LLM call, asking a specific binary question:

> *Does the summary contain any claims not present in the source?*

This is an **LLM-as-judge** pattern — using an AI model to evaluate another AI's
output against a defined rubric. The key design principle is **binary (pass/fail)**,
not a 1–5 scale. Binary forces you to define exactly what "good" means.

In [8]:
JUDGE_PROMPT = """You are an evaluation judge for AI-generated summaries of
industrial field reports. Your job is to check FAITHFULNESS — whether the
summary accurately represents the source report without adding information.

SOURCE REPORT:
\"\"\"{source}\"\"\"

GENERATED SUMMARY:
\"\"\"{summary}\"\"\"

TASK: Does the summary contain any claims, statistics, or findings that are
NOT present in the source report?

Answer in this EXACT format (two lines only):
VERDICT: YES or NO
EXPLANATION: [If YES, quote the specific claim that is not in the source.
If NO, state that all claims are supported by the source.]"""


def run_judge(source: str, summary: str) -> dict:
    """Run the LLM-as-judge faithfulness check."""
    prompt = JUDGE_PROMPT.format(source=source, summary=summary)

    resp = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    raw = resp.text.strip()

    # Parse the verdict
    verdict = None
    explanation = raw
    for line in raw.split("\n"):
        line_upper = line.strip().upper()
        if line_upper.startswith("VERDICT:"):
            verdict_text = line_upper.replace("VERDICT:", "").strip()
            if "YES" in verdict_text:
                verdict = "YES"
            elif "NO" in verdict_text:
                verdict = "NO"
        if line.strip().upper().startswith("EXPLANATION:"):
            explanation = line.strip()[len("EXPLANATION:"):].strip()

    # Faithfulness PASSES when there are NO unsupported claims
    passed = (verdict == "NO")

    return {
        "check": "llm_judge_faithfulness",
        "passed": passed,
        "verdict": verdict,
        "explanation": explanation,
        "raw_response": raw,
    }


# --- Run the judge ---
print("Running LLM-as-judge faithfulness check...")
print("(Sending source report + summary to Gemini for evaluation)\n")

judge_result = run_judge(SOURCE_REPORT, AI_SUMMARY)

status = "✅ Pass" if judge_result["passed"] else "❌ Fail"
print(f"Faithfulness: {status}")
print(f"Verdict:      {judge_result['verdict']} (unsupported claims found)")
print(f"Explanation:  {judge_result['explanation']}")
print(f"\nFull judge response:\n{judge_result['raw_response']}")

Running LLM-as-judge faithfulness check...
(Sending source report + summary to Gemini for evaluation)

Faithfulness: ❌ Fail
Verdict:      YES (unsupported claims found)
Explanation:  The claim "Downtime reduced by 23% compared to previous week" is not present in the source report.

Full judge response:
VERDICT: YES
EXPLANATION: The claim "Downtime reduced by 23% compared to previous week" is not present in the source report.


## Step 6 — Log to Langfuse (Optional)

If you provided Langfuse API keys in Step 1, this cell logs the entire
evaluation pipeline as a **trace** — the summary generation, each format check,
and the LLM judge call. You can then view the full pipeline in the Langfuse
dashboard.

If Langfuse is not enabled, this cell simply prints a message and moves on.

In [9]:
if LANGFUSE_ENABLED and langfuse is not None:
    try:
        # Create the outer trace for the full evaluation pipeline
        with langfuse.start_as_current_observation(
            as_type="span",
            name="nordwind-summary-evaluation",
        ) as trace_span:
            trace_span.update(
                input={"source_report": SOURCE_REPORT},
                output={"ai_summary": AI_SUMMARY},
                metadata={"site": "Hvide Sande", "date": "2026-02-17"},
            )

            # Log the (pre-written) summary as a generation
            with langfuse.start_as_current_observation(
                as_type="generation",
                name="summary-generation",
                model=GEMINI_MODEL,
            ) as gen_span:
                gen_span.update(
                    input=SOURCE_REPORT,
                    output=AI_SUMMARY,
                    metadata={"note": "Pre-written summary for demo (not a live LLM call)"},
                )

            # Log each format check as a span
            for r in format_results:
                with langfuse.start_as_current_observation(
                    as_type="span",
                    name=f"format-check-{r['check']}",
                ) as check_span:
                    check_span.update(
                        input={"check_type": r["check"]},
                        output={"passed": r["passed"], "detail": r["detail"]},
                    )
                    # Score each check
                    check_span.score(
                        name=f"format-{r['check']}",
                        value=1 if r["passed"] else 0,
                        data_type="BOOLEAN",
                        comment=r["detail"],
                    )

            # Log the LLM judge call as a generation
            with langfuse.start_as_current_observation(
                as_type="generation",
                name="llm-judge-faithfulness",
                model=GEMINI_MODEL,
            ) as judge_span:
                judge_prompt_filled = JUDGE_PROMPT.format(
                    source=SOURCE_REPORT, summary=AI_SUMMARY
                )
                judge_span.update(
                    input=judge_prompt_filled,
                    output=judge_result["raw_response"],
                )
                judge_span.score(
                    name="faithfulness",
                    value=1 if judge_result["passed"] else 0,
                    data_type="BOOLEAN",
                    comment=judge_result["explanation"],
                )

            # Score the overall trace
            overall_pass = all_format_passed and judge_result["passed"]
            trace_span.score_trace(
                name="overall-evaluation",
                value=1 if overall_pass else 0,
                data_type="BOOLEAN",
                comment="All checks passed" if overall_pass else "One or more checks failed",
            )

        langfuse.flush()

        # Get trace URL
        trace_id = langfuse.get_current_trace_id() if hasattr(langfuse, 'get_current_trace_id') else None
        base_url = os.environ.get('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')
        print("✅ Logged to Langfuse!")
        print(f"   View your traces at: {base_url}")
        print("   Navigate to Traces in the Langfuse dashboard to see the full pipeline.")

    except Exception as e:
        print(f"⚠️  Langfuse logging failed: {e}")
        print("   This doesn't affect the evaluation results above.")

else:
    print("ℹ️  Langfuse logging skipped (no API keys provided).")
    print("   All evaluation results are displayed in the cells above.")

✅ Logged to Langfuse!
   View your traces at: https://cloud.langfuse.com
   Navigate to Traces in the Langfuse dashboard to see the full pipeline.


## Step 7 — Consolidated Results

Here's the full evaluation summary in one view — the same information that would
appear in a Langfuse trace, but printed directly.

In [10]:
overall_pass = all_format_passed and judge_result["passed"]

print("Evaluation Results for NordWind Daily Summary (Hvide Sande, 2026-02-17)")
print("━" * 65)

for r in format_results:
    status = "✅ Pass" if r["passed"] else "❌ Fail"
    print(f"  Format: {r['check']:<18} {status}  ({r['detail']})")

judge_status = "✅ Pass" if judge_result["passed"] else "❌ Fail"
explanation_short = judge_result["explanation"][:80]
print(f"  LLM Judge: {'faithfulness':<14} {judge_status}  ({explanation_short})")

print("━" * 65)
overall_icon = "✅" if overall_pass else "❌"
print(f"  OVERALL: {overall_icon} {'PASS' if overall_pass else 'FAIL'}")
print()

if not judge_result["passed"]:
    print("💡 Key insight:")
    print("   This summary passed all format checks. It mentions the right")
    print("   turbines, the right date, and is within the word limit.")
    print()
    print("   A metric like ROUGE would also give it a reasonable score —")
    print("   most of the words overlap with what a good summary would say.")
    print()
    print("   But it contains a hallucinated statistic ('Downtime reduced by 23%')")
    print("   that appears nowhere in the source report. Only a faithfulness-focused")
    print("   evaluation — like an LLM judge — can catch this kind of error.")

Evaluation Results for NordWind Daily Summary (Hvide Sande, 2026-02-17)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Format: word_count         ✅ Pass  (46 words (limit: 60))
  Format: turbine_id         ✅ Pass  (Found: NW-023, NW-024)
  Format: date_present       ✅ Pass  (Matched: 17 Feb 2026)
  LLM Judge: faithfulness   ❌ Fail  (The claim "Downtime reduced by 23% compared to previous week" is not present in )
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OVERALL: ❌ FAIL

💡 Key insight:
   This summary passed all format checks. It mentions the right
   turbines, the right date, and is within the word limit.

   A metric like ROUGE would also give it a reasonable score —
   most of the words overlap with what a good summary would say.

   But it contains a hallucinated statistic ('Downtime reduced by 23%')
   that appears nowhere in the source report. Only a faithfulness-focused
   evaluation — like an LLM judge — can catch this kind of erro

## Step 8 — Try It Yourself (Optional Extension)

Generate a **new** summary from the source report using Gemini, then run the
full evaluation pipeline on it. Does Gemini hallucinate when writing its own
summary? Or does it produce a faithful one?

In [11]:
# Generate a fresh summary from the source report
SUMMARY_PROMPT = """You are writing a daily operations summary for a wind energy
company's management team. Summarise the following field report in 2-3 sentences.
Include only facts stated in the report. Do not add statistics, comparisons,
or conclusions not present in the source.

Field Report:
\"\"\"{report}\"\"\"

Summary:"""

print("Generating a new summary from the source report...\n")
resp = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=SUMMARY_PROMPT.format(report=SOURCE_REPORT),
)
new_summary = resp.text.strip()
print(f"Generated summary:\n{new_summary}\n")
print(f"({len(new_summary.split())} words)")

# --- Run the full evaluation pipeline on the new summary ---
print("\n" + "=" * 65)
print("EVALUATING THE NEW SUMMARY")
print("=" * 65 + "\n")

# Format checks
new_format_results = [
    check_word_count(new_summary),
    check_turbine_ids(new_summary, expected_ids=["NW-023", "NW-024"]),
    check_date_present(new_summary),
]

for r in new_format_results:
    status = "✅ Pass" if r["passed"] else "❌ Fail"
    print(f"  Format: {r['check']:<18} {status}  ({r['detail']})")

# LLM judge
print("\n  Running LLM judge...")
new_judge = run_judge(SOURCE_REPORT, new_summary)
judge_status = "✅ Pass" if new_judge["passed"] else "❌ Fail"
print(f"  LLM Judge: {'faithfulness':<14} {judge_status}")
print(f"  Explanation: {new_judge['explanation']}")

new_overall = all(r["passed"] for r in new_format_results) and new_judge["passed"]
print(f"\n  OVERALL: {'✅ PASS' if new_overall else '❌ FAIL'}")

if new_judge["passed"]:
    print("\n💡 The model produced a faithful summary this time!")
    print("   But run this cell a few more times — LLM outputs are probabilistic,")
    print("   so the same model might hallucinate on the next run.")

Generating a new summary from the source report...

Generated summary:
Technicians completed inspections on turbines NW-023 and NW-024 and replaced a worn mooring line on the vessel Havørn. While turbine NW-024 remains nominal, NW-023 showed elevated pitch bearing vibrations requiring a follow-up inspection. Site operations were paused for 45 minutes due to high wave conditions, though no personnel safety incidents occurred.

(51 words)

EVALUATING THE NEW SUMMARY

  Format: word_count         ✅ Pass  (51 words (limit: 60))
  Format: turbine_id         ✅ Pass  (Found: NW-023, NW-024)
  Format: date_present       ❌ Fail  (Date not found in summary)

  Running LLM judge...
  LLM Judge: faithfulness   ✅ Pass
  Explanation: All claims in the summary are supported by the source report.

  OVERALL: ❌ FAIL

💡 The model produced a faithful summary this time!
   But run this cell a few more times — LLM outputs are probabilistic,
   so the same model might hallucinate on the next run.


## Discussion Questions

1. **Why binary pass/fail?** The judge prompt asks "does this contain unsupported
   claims — YES or NO." Why is this better than "rate faithfulness 1–5"?
   *(Hint: what's the difference between a 3 and a 4?)*

2. **Format checks are necessary but not sufficient.** The pre-written summary
   passed every format check. What does this tell you about relying solely on
   automated structural validation?

3. **Who should design the judge prompt?** The faithfulness rubric was written by
   someone who understands what matters in a NordWind field report. Could an
   engineer without domain knowledge write an equally effective rubric?

4. **What about the judge's own errors?** The LLM judge is also an AI — it could
   miss a hallucination, or flag something that's actually in the source. How would
   you evaluate the evaluator?

## Connection to the Slides

This notebook demonstrates the pipeline shown in **Slides 32–37**:

| Slide | What it shows | Notebook cell |
|-------|--------------|---------------|
| Slide 33 | Source field report | Step 2 |
| Slide 34 | AI summary + format checks | Steps 3–4 |
| Slide 35 | LLM judge catches hallucination | Step 5 |
| Slide 36 | Langfuse dashboard trace | Step 6 |
| Slide 37 | The punchline — why you need both | Step 7 |